# 27 | LangChain Middleware：dynamic_prompt动态切换提示词

有些 Agent 不是永远只干一件事。

同一个“文档助手”，用户可能让它写周报，也可能让它写项目复盘。如果系统提示词写死，模型就容易用同一种语气处理所有任务：周报写得像事故复盘，复盘写得像朋友圈总结，读起来会有点尴尬。

`@dynamic_prompt` 解决的就是这个问题：**每次模型调用前，根据当前任务动态决定系统提示词。**

## 一、它和 before_model 不一样

`before_model` 更适合做日志、检查、更新 state。

`dynamic_prompt` 更专注：返回本轮要用的系统提示词。

它拿到的是 `ModelRequest`，可以看：

- `request.messages`：当前要发给模型的消息；
- `request.state`：Agent 当前状态；
- `request.runtime.context`：本次运行传入的业务上下文；
- `request.model`：本次准备调用的模型。

但它的目标不是执行模型，也不是拦截工具，而是回答一个问题：**这次该用哪段提示词？**

In [ ]:
import os
from typing import TypedDict

from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, dynamic_prompt
from langchain.chat_models import init_chat_model

load_dotenv(dotenv_path=".env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
)

## 二、场景：一个文档助手，三种写法

这里不做复杂工具，只看提示词切换。

用户输入里出现“周报”，就用周报提示词；出现“复盘”或“事故”，就用复盘提示词；其他情况，就用普通文档整理提示词。

另外，用 `runtime.context` 传一个 `audience`，表示这份内容主要给谁看。它不是用户消息，而是系统侧传进来的业务背景。

In [ ]:
class RunContext(TypedDict):
    # audience 是本次运行的业务上下文，比如“研发负责人”“运营团队”。
    # 它不会自动变成用户消息，但 dynamic_prompt 可以读取它，用来生成更合适的系统提示词。
    audience: str


def latest_user_text(request: ModelRequest) -> str:
    """取最近一条用户消息，方便判断当前任务类型。"""
    for message in reversed(request.messages):
        if message.type == "human":
            return str(message.content)
    return ""

In [ ]:
used_prompts = []


@dynamic_prompt
def report_prompt_switch(request: ModelRequest) -> str:
    user_text = latest_user_text(request)
    audience = (request.runtime.context or {}).get("audience", "内部同事")

    if "复盘" in user_text or "事故" in user_text:
        prompt = (
            "你是一个严谨的项目复盘助手。"
            f"这份内容主要给{audience}看。"
            "请按：背景、影响、原因、处理过程、后续改进 来组织。"
            "语气要客观，不甩锅，也不要写成鸡汤。"
        )
        print("[dynamic_prompt] 使用：项目复盘提示词")
        used_prompts.append(prompt)
        return prompt

    if "周报" in user_text:
        prompt = (
            "你是一个专业的工作周报助手。"
            f"这份内容主要给{audience}看。"
            "请按：本周完成、关键进展、风险问题、下周计划 来组织。"
            "表达要简洁，重点突出，不要像流水账。"
        )
        print("[dynamic_prompt] 使用：周报提示词")
        used_prompts.append(prompt)
        return prompt

    prompt = (
        "你是一个专业的文档整理助手。"
        f"这份内容主要给{audience}看。"
        "请把内容整理得清楚、准确、可执行。"
    )
    print("[dynamic_prompt] 使用：普通文档整理提示词")
    used_prompts.append(prompt)
    return prompt

这段代码的重点不是关键词判断，而是 `return`。

`used_prompts` 只是为了演示时把本轮系统提示词打印出来，真实项目里不一定要这样存。

`dynamic_prompt` 返回的字符串，会被 LangChain 设置成本轮模型调用的系统提示词。

也就是说，同一个 Agent，不同输入，可以让模型进入不同工作模式。

In [ ]:
agent = create_agent(
    model=model,
    middleware=[report_prompt_switch],
    context_schema=RunContext,
)

questions = [
    "帮我把这段内容整理成周报：本周完成订单导出接口开发，修复两个权限问题，下周准备接入异步任务。",
    "帮我把这件事整理成复盘：昨晚订单导出任务超时，影响财务下载月度报表，后来通过拆分任务临时恢复。",
]

used_prompts.clear()

for question in questions:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        context={"audience": "研发负责人"},
    )

    print("\n[用户问题]")
    print(question)
    print("\n[本轮系统提示词]")
    print(used_prompts[-1])
    print("\n[模型回复]")
    print(result["messages"][-1].content)
    print("-" * 60)

## 三、输出怎么看

运行后，你会先看到类似日志：

```text
[dynamic_prompt] 使用：周报提示词
[dynamic_prompt] 使用：项目复盘提示词
```

同时还会打印 `[本轮系统提示词]`，你可以直接看到周报和复盘两次调用使用的是不同提示词。

如果不用 `dynamic_prompt`，通常只能写一个很大的通用 prompt，把周报、复盘、说明文、公众号风格全塞进去。能不能用？能。就是 prompt 越写越像菜单，模型点错菜的概率也会变高。

## 四、什么时候该用 dynamic_prompt

适合用它的情况：

- 同一个 Agent 要处理多类任务；
- 不同用户角色需要不同回答风格；
- 同一个问题要按不同受众输出，比如给老板、研发、运营看的版本不同；
- 希望系统提示词根据上下文变化，而不是写死在 `create_agent(system_prompt=...)` 里。

不太适合的情况：

- 只是想记录日志，用 `before_model`；
- 想重试、切模型、兜底，用 `wrap_model_call`；
- 想拦截工具调用，用 `wrap_tool_call`。

`dynamic_prompt` 管的是“模型这次应该用什么身份和规则思考”。别让它顺手包办所有 Middleware 的活，容易累，也容易乱。

## 五、和其他 Middleware 的区别

| 你想做什么 | 更适合用 |
|---|---|
| 动态决定系统提示词 | `dynamic_prompt` |
| 模型调用前记录消息数量 | `before_model` |
| 模型返回后统计工具调用 | `after_model` |
| 本地模型失败后切云端模型 | `wrap_model_call` |
| 工具执行前做审批或拦截 | `wrap_tool_call` |

记住这个判断：

**如果目标是改变模型这次看到的系统提示词，优先考虑 `dynamic_prompt`。**

## 小结

`@dynamic_prompt` 是一个专门用来动态生成系统提示词的 Middleware。

它接收 `ModelRequest`，可以根据当前消息、state、runtime context 等信息，返回本轮模型调用要使用的提示词。

它适合做任务路由、角色切换、受众切换、输出风格切换。

一个写死的 prompt 像一件万能外套，什么场合都能穿，但不一定合身。`dynamic_prompt` 的价值，就是让 Agent 在不同场合换对衣服。别浮夸，合身就行。